# 03 — Data Preparation (CRISP-DM)

Materializa a **base perfeita** para modelagem, com três garantias:

1. **Fail-closed**: a base só avança se passar na validação de esquema/regras.
2. **Anti-leakage**: colunas agregadas do próprio dia-alvo são explicitamente excluídas.
3. **Split temporal estrito**: treino vê o passado; teste cobre o futuro (Q4).

Tudo em código testável (`src/data/preparar_dataset.py`), nunca em célula solta.


## 1) Validação fail-closed da base bruta

In [1]:
import json

from src.data import preparar_dataset as prep

bruto = prep.carregar_dados_brutos()
problemas = prep.validar_esquema(bruto)
assert problemas == [], problemas
print(f"Base bruta validada: {bruto.shape[0]:,} linhas x {bruto.shape[1]} colunas")

Base bruta validada: 35,088 linhas x 38 colunas


## 2) Limpeza + engenharia de features

In [2]:
limpo = prep.limpar_dados(bruto)
enriquecido = prep.engenharia_features(limpo)
list(enriquecido.columns)

['data',
 'loja',
 'produto',
 'categoria',
 'custo_producao',
 'preco_original',
 'desconto_pct',
 'preco_final',
 'quantidade_vendida_dia',
 'dia_semana',
 'mes',
 'feriado_flag',
 'evento_especial',
 'promocao_flag',
 'promocao_tipo',
 'promocao_inicio',
 'promocao_fim',
 'canal',
 'regiao',
 'bairro',
 'perfil_loja',
 'marca',
 'tamanho_embalagem',
 'tipo_produto',
 'preco_competidor',
 'indice_preco_competidor',
 'gasto_marketing',
 'campanha_id',
 'estoque_disponivel',
 'ruptura_flag',
 'lead_time_reposicao',
 'temperatura',
 'chuva_mm',
 'custo_logistico',
 'custo_marketing_unitario',
 'preco_minimo',
 'margem_minima',
 'quantidade_vendida_mes',
 'receita_dia',
 'lucro_dia',
 'margem_unitaria',
 'ano',
 'trimestre',
 'dia_semana_idx',
 'fim_de_semana']

## 3) Guardrail de vazamento

`quantidade_vendida_mes` agrega o próprio dia-alvo. Ela permanece na base para
relatórios, mas é **excluída** das features.

In [3]:
print("Colunas com vazamento:", prep.colunas_com_vazamento())
features = prep.selecionar_features(enriquecido)
print(f"Nº de features leakage-safe: {len(features)}")
assert "quantidade_vendida_mes" not in features

Colunas com vazamento: ('quantidade_vendida_mes',)
Nº de features leakage-safe: 39


## 4) Split temporal estrito

In [4]:
preparado = prep.marcar_split_temporal(enriquecido)
preparado["split"].value_counts()

split
treino       26256
validacao     4416
teste         4416
Name: count, dtype: int64

In [5]:
treino_max = preparado.loc[preparado["split"] == "treino", "data"].max()
teste_min = preparado.loc[preparado["split"] == "teste", "data"].min()
print("Último dia do treino:", treino_max.date())
print("Primeiro dia do teste:", teste_min.date())
assert treino_max < teste_min, "Vazamento temporal: futuro dentro do treino."


Último dia do treino: 2025-06-30
Primeiro dia do teste: 2025-10-01


## 5) Resumo de qualidade e persistência (governança via DVC)

In [6]:
print(json.dumps(prep.resumo_qualidade(preparado), ensure_ascii=False, indent=2, default=str))

{
  "linhas": 35088,
  "colunas": 46,
  "periodo_inicio": "2024-01-01",
  "periodo_fim": "2025-12-31",
  "lojas": [
    "Unidade A",
    "Unidade B"
  ],
  "produtos": 24,
  "receita_total": 7618719.4,
  "nulos_totais": 0,
  "nulos_intencionais": 61308,
  "split_contagem": {
    "treino": 26256,
    "validacao": 4416,
    "teste": 4416
  }
}


In [7]:
destino = prep.salvar_parquet(preparado)
print("Base preparada salva em:", destino)

Base preparada salva em: C:\Users\flavi\Documents\GitHub\Otimizacao_de_Receita\dados\dataset_cafeterias_rj_preparado.parquet


## Conclusão

A base preparada (`dataset_cafeterias_rj_preparado.parquet`) é o **insumo canônico**
das fases seguintes: contém as features leakage-safe, os alvos de negócio e a coluna
`split` que garante a avaliação honesta no futuro. É governada pelo DVC, nunca pelo Git.
